# Question 7

In [0]:
create volume cyntexa_dev.sales.my_volume2

In [0]:
-- create or replace table cyntexa_dev.sales.sales_diff_schema as 
-- select * , _metadata.file_path as file_name , _metadata.file_modification_time as ingestion_timestamp
--  from read_files('/Volumes/cyntexa_dev/sales/my_volume2/*.csv' ,
--  Delimiter=',',
--  header=True
--  )

CREATE OR REPLACE TABLE cyntexa_dev.sales.sales_diff_schema AS 
SELECT 
*
FROM read_files(
  '/Volumes/cyntexa_dev/sales/my_volume2/*.csv',
  format => 'csv',
  -- Delimiter=',',
  header => true,
  schema => 'sale_id INT, product STRING, amount DOUBLE', -- Explicit Strict Schema
  rescuedDataColumn => '_rescued_data' -- Enable rescued data column
);





In [0]:
desc cyntexa_dev.sales.sales_diff_schema


In [0]:
select* from cyntexa_dev.sales.sales_diff_schema

# Handling Format Variations & Detecting Schema Mismatches in Databricks(`read_files`)

## 1. Handling Varying File Formats using `read_files()`

### Different Delimiters
Standard CSV uses commas. For files separated by pipe (`|`), tab (`\t`), or semi-colon (`;`), pass the explicit `delimiter` option:

```sql
SELECT * FROM read_files(
  '/Volumes/cyntexa_dev/sales/my_volume/*.txt',
  format => 'csv',
  delimiter => '|',
  header => true
);
```

### Extra Columns Handling
* **Schema Evolution (To automatically add new columns):**
  Omit the `schema` parameter. Auto Loader dynamically merges schemas, creating new columns in the target table and populating `NULL` for files where those fields are absent.
* **Fixed Table Structure (Strict Schema):**
  Provide an explicit `schema` parameter (`schema => 'sale_id INT, product STRING, amount DOUBLE'`).

---

## 2. Detecting Mismatches Before Downstream Breakage

To prevent invalid data types or malformed rows from silently breaking downstream reports, combine **Strict Schema Enforcement** with the **`_rescued_data` Column**:

* **Data Type Mismatches:** If a column contains invalid types (e.g., `"FREE"` in a numeric `amount` column), the target field receives `NULL`, and the raw value is saved in `_rescued_data` as JSON.
* **Unexpected Columns:** Under a strict schema, extra columns do not alter table definitions; their data is safely redirected to `_rescued_data`.

---

## 3. Pre-Downstream Validation Gate

Run an automated audit query immediately after the ingestion step. If the bad record count is greater than zero, fail the pipeline job or raise an alert before downstream gold tables execute:

```sql
SELECT 
  _metadata.file_name AS source_file,
  COUNT(1) AS bad_records_count
FROM cyntexa_dev.sales.sales_strict_test
WHERE _rescued_data IS NOT NULL
GROUP BY _metadata.file_name;

# Question 8

# Strategic Decision Memo

**To:** Engineering Leadership & Data Architecture Board  
**From:** Principal Data Architect  
**Subject:** Table Format Selection Framework: When to Choose Apache Iceberg (or Delta UniForm) over Native Delta Lake  

---

## 1. Executive Summary

As Cyntexa expands its modern data stack, maintaining high query performance while ensuring multi-engine interoperability across **Databricks, Snowflake, and Trino** is critical. While native **Delta Lake** remains the default choice for pure Databricks/Spark-centric workloads, lock-in risks and cross-engine compute costs dictate the need for a standardized selection framework.

This memo defines **when and why Cyntexa should choose Apache Iceberg or Delta Universal Format (UniForm) over native Delta Lake**.

```
                           [ Table Requirements ]
                                      |
         +----------------------------+----------------------------+
         |                                                         |
[ Pure Spark/Databricks ]                               [ Multi-Engine / Hybrid ]
 (High streaming/Write-heavy)                           (Snowflake, Trino, Starburst)
         |                                                         |
   (Native Delta)                               +------------------+------------------+
                                                |                                     |
                                   [ Databricks is Primary ]             [ Multi-Engine First ]
                                    (Reads by Snowflake/Trino)            (No Databricks dependency)
                                                |                                     |
                                         (Delta UniForm)                       (Native Iceberg)
```

---

## 2. Architectural Comparison: Delta vs. Iceberg vs. Delta UniForm

While both formats use Parquet for storage and provide ACID transactions, time travel, and schema evolution, their core architectures differ:

| Feature | Native Delta Lake | Apache Iceberg | Delta UniForm |
| :--- | :--- | :--- | :--- |
| **Transaction Model** | File-based append-only JSON transaction log (`_delta_log`). | Hierarchical snapshots (`metadata.json` $\rightarrow$ Manifest Lists $\rightarrow$ Manifests). | Writes native Delta log + asynchronously generates Iceberg metadata. |
| **Primary Engine** | Apache Spark / Databricks | Engine-agnostic (Trino, Snowflake, Flink, Starburst). | Databricks (Writer) $\rightarrow$ Multi-Engine (Reader). |
| **Schema & Partition Evolution** | Strict schema enforcement; partition changes require data rewrite. | In-place partition evolution (hidden partitioning) without rewriting data. | Inherits Delta schema rules; translates evolution to Iceberg metadata. |
| **Downstream Compatibility** | High with Spark; requires connectors or legacy manifest generation for Trino/Snowflake. | Native catalog support across Snowflake, Trino, Dremio, and BigQuery. | Native read capabilities for any engine supporting Apache Iceberg REST catalog specs. |

---

## 3. Tool Interoperability & Downstream Impact

### 3.1 Snowflake Integration
* **Native Delta:** Requires external tables via manifest files or external pointers. Query performance often degraded due to lack of advanced file pruning.
* **Apache Iceberg:** First-class citizen. Snowflake can query Iceberg tables via **Iceberg Catalog (REST / Glue / Polaris)** or even manage them as Snowflake-managed Iceberg Tables (using Snowflake compute for maintenance).
* **Delta UniForm:** Snowflake reads UniForm tables as standard Iceberg tables. No manifest generation required.

### 3.2 Trino / Starburst Integration
* **Native Delta:** Supported via the Trino Delta Lake connector, but metadata reads can lag when transaction logs are large.
* **Apache Iceberg:** Native, highly optimized connector. Trino uses Iceberg manifest files for fine-grained partition and file pruning, offering superior query execution times.
* **Delta UniForm:** Trino queries the generated Iceberg metadata directly via the Iceberg connector, bypassing Spark/Delta-specific log parsing.

---

## 4. Decision Matrix: When to Select Which Format

### Use Native Delta Lake when:
1. **Pipeline is 100% Databricks-Native:** Reads and writes are isolated to Spark/Databricks ecosystems.
2. **High-Frequency Real-Time Streaming:** The workload relies on Structured Streaming with frequent micro-batch writes where metadata translation overhead is unwanted.
3. **Databricks-Exclusive Features are Required:** Heavy reliance on Liquid Clustering, Deletion Vectors, or native Databricks Unity Catalog features without external consumption needs.

### Choose Delta UniForm when:
1. **Databricks is the Primary ETL Engine, but Snowflake/Trino require Read Access:** Enables writing in native Delta while granting external engines zero-copy access via Iceberg metadata.
2. **Standardization without Migration:** Cyntexa teams already have pipelines built on Delta Lake but face integration friction with clients using Snowflake or Trino.
3. **Low-Maintenance Dual Compatibility:** Eliminates the operational overhead of running double-write pipelines or maintaining manual manifest files.

### Choose Native Apache Iceberg when:
1. **Multi-Engine Write Architecture:** The data stack uses engines other than Spark (e.g., **Trino for ad-hoc SQL, Flink for real-time streaming, Snowflake for analytics**) to write directly into the table.
2. **Independent Data Architecture (No Vendor Lock-in):** The table must exist independently of Databricks (e.g., managed via an open REST catalog like Tabular/Polaris or AWS Glue).
3. **Advanced Partitioning Needs:** The dataset requires **partition evolution** (changing daily partition to hourly without rewriting historic partitions) or **hidden partitioning**.

---

## 5. Summary Strategy for Cyntexa

```
    Workload Type              Recommended Format           Primary Reason
----------------------------------------------------------------------------------------
Spark ETL -> Databricks Only   Native Delta Lake            Maximum write performance & native features.
Spark ETL -> Snowflake/Trino   Delta UniForm                Zero-copy Iceberg reads without changing write pipeline.
Multi-Engine / Non-Spark Writes Native Apache Iceberg       Native interoperability across engines & open catalogs.
```

# Question 9 

In [0]:
create or replace table cyntexa_dev.sales.sales_diff_schema_lineage as 
select * , _metadata.file_path as file_name , _metadata.file_modification_time as modification_time
,current_timestamp() as ingestion_timestamp
from read_files('/Volumes/cyntexa_dev/sales/my_volume2/sales_day1.csv')

In [0]:
COPY INTO cyntexa_dev.sales.sales_diff_schema_lineage
FROM (
  SELECT 
    CAST(sale_id AS INT) AS sale_id,
    CAST(product AS STRING) AS product,
    CAST(amount AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS _rescued_data,
    _metadata.file_path AS file_name,
    _metadata.file_modification_time AS modification_time,
    current_timestamp() AS ingestion_timestamp
  FROM '/Volumes/cyntexa_dev/sales/my_volume2/sales_day2_bad.csv'
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'false');

In [0]:
select * from  cyntexa_dev.sales.sales_diff_schema_lineage

In [0]:
DESCRIBE HISTORY cyntexa_dev.sales.sales_diff_schema_lineage;

In [0]:
select * from  cyntexa_dev.sales.sales_diff_schema_lineage  version as of 3
where amount<0

In [0]:
restore table cyntexa_dev.sales.sales_diff_schema_lineage to version as of 2

In [0]:
select * from cyntexa_dev.sales.sales_diff_schema_lineage where amount<0